# Projeto 1: Limpeza dataset ecommerce

## Importação das bibliotecas


In [1]:
# Bibliotecas para validação/formatação de telefone e e-mail
# (rode esta célula uma vez caso ainda não tenha as bibliotecas instaladas)
%pip install phonenumbers email-validator --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import phonenumbers
from phonenumbers import NumberParseException
from email_validator import validate_email, EmailNotValidError

## Importação dos dados

In [3]:

importacao=pd.read_csv('vendas_ecommerce_dados_brutos.csv')
df=importacao.copy()

## Entendendo os dados


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_pedido          504 non-null    int64 
 1   data_pedido        501 non-null    object
 2   id_cliente         501 non-null    object
 3   nome_cliente       481 non-null    object
 4   email              483 non-null    object
 5   telefone           444 non-null    object
 6   idade              481 non-null    object
 7   cidade             486 non-null    object
 8   estado             492 non-null    object
 9   categoria_produto  500 non-null    object
 10  produto            502 non-null    object
 11  quantidade         490 non-null    object
 12  preco_unitario     490 non-null    object
 13  valor_total        472 non-null    object
 14  forma_pagamento    493 non-null    object
 15  status_pedido      501 non-null    object
 16  avaliacao_cliente  418 non-null    object
dt

## Tratando colunas


In [5]:
#Tratando colunas numéricas float

#Limpando os valores inválidos e mantendo somente os dados numéricos e os separadores de decimais e de milhar
for col in ['valor_total','preco_unitario']:
    df[col]=df[col].astype('str').str.replace(r'[^\d\.\,]','',regex=True).str.strip()

#lidando com as colunas numéricas float despadronizadas, algumas usavam ',' como separador de decimal e outras usavam '.' oq dificultava em criar uma solução geral para a coluna 

for col in ['valor_total','preco_unitario']:
    virgula= df[col].str.contains(r',\d{2}$',regex=True)
    df.loc[virgula,col] = (df.loc[virgula,col].str.replace('.','')
                            .str.replace(',','.')
                            .str.strip())
    

for col in ['valor_total','preco_unitario']:
    df.loc[df[col]=='', col] = np.nan
    df[col]=df[col].astype('float')


In [6]:
# essas linhas servem para lidar com os caracteres que não são dígitos nas respectivas colunas
df['avaliacao_cliente']=df['avaliacao_cliente'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade']=df['quantidade'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade'].value_counts()


quantidade
1    173
2    124
3     80
4     57
5     41
0      9
       6
Name: count, dtype: int64

In [7]:
#trocando os espaços por nulos para poder criar colunas numéricas e imputar valores
for col in ['quantidade','avaliacao_cliente']:
   df.loc[df[col]=='',col] = np.nan

# A conversão para int não funciona por haver nulos
df['quantidade']=df['quantidade'].astype('float')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('float')

#imputando a mediana aos nulos
for col in ['quantidade','preco_unitario','avaliacao_cliente']:
   df[col]=df[col].fillna((df[col].median(skipna=True)))

#conversão para int
df['quantidade']=df['quantidade'].astype('Int64')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('Int64')

In [8]:
df['valor_total']=df['valor_total'].fillna((df['preco_unitario']*df['quantidade']))
df[['valor_total', 'preco_unitario', 'quantidade']].isna().sum()

valor_total       0
preco_unitario    0
quantidade        0
dtype: int64

In [9]:

df['id_cliente']=df['id_cliente'].astype('str').str.replace(r'\D','',regex=True).str.strip()
df.loc[df['id_cliente']=='','id_cliente'] = np.nan

# Exclui registros sem id_cliente, pois sem esse identificador
# não é possível relacionar o cliente a consultas posteriores.
df = df.dropna(subset=['id_cliente'])

In [10]:
#retirando não dígitos
df['idade']=df['idade'].str.replace(r'\D','',regex=True)
df['idade']=df['idade'].str.strip()
df.loc[df['idade']=='','idade'] = np.nan

# O int padrão python/numpy não aceita nulos, diferente do Int do pandas (o float padrão aceita também).
df['idade']=df['idade'].astype('Int64')


In [11]:
#imputando nulos para a coluna de idade
# essas condição normalmente iria depender das regras de negócio, então
df.loc[df['idade']<18,'idade'] = df['idade'].median()
df['idade']=df['idade'].fillna((df['idade'].median()))
df.loc[df['idade']>80,'idade'] = df['idade'].median()


In [12]:
#trocando os separadores das datas
df['data_pedido']=df['data_pedido'].str.replace('-','/')

#formatando a coluna para o tipo data
df['data_pedido']=pd.to_datetime(df['data_pedido'],format='mixed')

In [13]:
#lidando com caracteres indesejados 
df['nome_cliente']=df['nome_cliente'].str.replace(r'[^\w\s\.]','',regex=True).str.title().str.strip()
df.loc[df['nome_cliente']=='','nome_cliente'] = np.nan

df.info()
df


<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 0 to 503
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_pedido          500 non-null    int64         
 1   data_pedido        500 non-null    datetime64[ns]
 2   id_cliente         500 non-null    object        
 3   nome_cliente       474 non-null    object        
 4   email              481 non-null    object        
 5   telefone           444 non-null    object        
 6   idade              500 non-null    Int64         
 7   cidade             484 non-null    object        
 8   estado             491 non-null    object        
 9   categoria_produto  500 non-null    object        
 10  produto            500 non-null    object        
 11  quantidade         500 non-null    Int64         
 12  preco_unitario     500 non-null    float64       
 13  valor_total        500 non-null    float64       
 14  forma_pagamento

,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente
0,1041,2023-07-26,306,Renan Ribeiro,leonardo88@example.org,NaN,59,Florianópolis,SC,roupas,Camiseta Básica,2,49.9,99.8,CARTÃO DE DÉBITO,Devolvido,5
1,1047,2024-03-27,195,Diego Melo,novaisdavi@example.com,+55 11 97309-1356,70,CONTAGEM,MG,Roupas,Calça Jeans,4,129.9,519.6,Cartão de Débito,processando,10
2,1344,2023-10-16,165,Nicole Moreira,liviacaldeira@example.com,94400-6566,49,Porto Alegre,Rio Grande do Sul,Esporte e Lazer,Bola de Futebol Oficial,4,89.9,359.6,boleto bancário,CANCELADO,1
3,1137,2025-08-01,45,Fernanda Freitas,ucorreia@example.com,NaN,41,Olinda,PE,Roupas,Vestido Casual,3,159.9,479.7,boleto bancário,Em Transporte,1
4,1110,2023-08-27,149,Cecilia Novaes,ryanborges@example.org,(85) 94156-8143,29,Petrópolis,RJ,Roupas,Calça Jeans,2,129.9,259.8,Cartão de Crédito,Cancelado,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,1007,2024-12-25,109,Esther Freitas,novaesdavi-lucas@example.net,(85) 93142-9713,67,SANTOS,SP,Roupa,Jaqueta Corta-Vento,3,199.9,599.7,Cartão de Débito,DEVOLVIDO,4
500,1221,2024-03-17,265,Luiz Felipe Melo,nascimentopedro@example.net,(11) 97344-8413,59,NaN,RS,ELETRÔNICOS,Notebook,2,3299.0,6598.0,BOLETO BANCÁRIO,devolvido,5
501,1462,2025-08-22,139,Davi Miguel Freitas,guerrarenan@example.org,79 99027-9526,36,Duque de Caxias,RJ,Roupas,Vestido Casual,2,159.9,319.8,Pix,CANCELADO,5
502,1300,2024-02-24,319,NaN,ravi-luccadas-neves@example.net,(71) 96635-4587,45,Parintins,Amazonas,Casa e Decoração,Tapete Sala,4,249.9,999.6,pix,Processando,4


In [14]:
#Tratando a coluna de telefone com a biblioteca phonenumbers
#Interpreta o número assumindo Brasil como região padrão, valida e formata no padrão nacional

def limpar_telefone(tel):
    if pd.isna(tel):
        return np.nan
    tel = re.sub(r'[^\d+]', '', str(tel))
    if tel == '':
        return np.nan
    try:
        numero = phonenumbers.parse(tel, 'BR')
        if phonenumbers.is_valid_number(numero):
            return phonenumbers.format_number(numero, phonenumbers.PhoneNumberFormat.NATIONAL)
        return np.nan
    except NumberParseException:
        return np.nan

df['telefone'] = df['telefone'].apply(limpar_telefone)
df

,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente
0,1041,2023-07-26,306,Renan Ribeiro,leonardo88@example.org,NaN,59,Florianópolis,SC,roupas,Camiseta Básica,2,49.9,99.8,CARTÃO DE DÉBITO,Devolvido,5
1,1047,2024-03-27,195,Diego Melo,novaisdavi@example.com,(11) 97309-1356,70,CONTAGEM,MG,Roupas,Calça Jeans,4,129.9,519.6,Cartão de Débito,processando,10
2,1344,2023-10-16,165,Nicole Moreira,liviacaldeira@example.com,NaN,49,Porto Alegre,Rio Grande do Sul,Esporte e Lazer,Bola de Futebol Oficial,4,89.9,359.6,boleto bancário,CANCELADO,1
3,1137,2025-08-01,45,Fernanda Freitas,ucorreia@example.com,NaN,41,Olinda,PE,Roupas,Vestido Casual,3,159.9,479.7,boleto bancário,Em Transporte,1
4,1110,2023-08-27,149,Cecilia Novaes,ryanborges@example.org,(85) 94156-8143,29,Petrópolis,RJ,Roupas,Calça Jeans,2,129.9,259.8,Cartão de Crédito,Cancelado,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,1007,2024-12-25,109,Esther Freitas,novaesdavi-lucas@example.net,(85) 93142-9713,67,SANTOS,SP,Roupa,Jaqueta Corta-Vento,3,199.9,599.7,Cartão de Débito,DEVOLVIDO,4
500,1221,2024-03-17,265,Luiz Felipe Melo,nascimentopedro@example.net,(11) 97344-8413,59,NaN,RS,ELETRÔNICOS,Notebook,2,3299.0,6598.0,BOLETO BANCÁRIO,devolvido,5
501,1462,2025-08-22,139,Davi Miguel Freitas,guerrarenan@example.org,(79) 99027-9526,36,Duque de Caxias,RJ,Roupas,Vestido Casual,2,159.9,319.8,Pix,CANCELADO,5
502,1300,2024-02-24,319,NaN,ravi-luccadas-neves@example.net,(71) 96635-4587,45,Parintins,Amazonas,Casa e Decoração,Tapete Sala,4,249.9,999.6,pix,Processando,4


In [15]:
for col in ['forma_pagamento','status_pedido','categoria_produto','cidade']:
    df[col]=df[col].astype('str').str.strip().str.title()
    df[col]=df[col].str.replace(r'[^\w\s]','',regex=True)
    df.loc[df[col]=='',col] = np.nan
    
df['produto']=df['produto'].astype('str').str.strip().str.title()
df['produto']=df['produto'].str.replace(r'[^\w\s]','',regex=True)
df.loc[df['produto']=='','produto'] = np.nan

In [ ]:
#Valida o formato do endereço, normaliza e descarta valores inválidos como nulos

def limpar_email(email):
    if pd.isna(email):
        return np.nan
    try:
        resultado = validate_email(str(email).strip(), check_deliverability=False)
        return resultado.normalized.lower()
    except EmailNotValidError:
        return np.nan

df['email'] = df['email'].apply(limpar_email)
df

,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente
0,1041,2023-07-26,306,Renan Ribeiro,leonardo88@example.org,NaN,59,Florianópolis,SC,Roupas,Camiseta Básica,2,49.9,99.8,Cartão De Débito,Devolvido,5
1,1047,2024-03-27,195,Diego Melo,novaisdavi@example.com,(11) 97309-1356,70,Contagem,MG,Roupas,Calça Jeans,4,129.9,519.6,Cartão De Débito,Processando,10
2,1344,2023-10-16,165,Nicole Moreira,liviacaldeira@example.com,NaN,49,Porto Alegre,Rio Grande do Sul,Esporte E Lazer,Bola De Futebol Oficial,4,89.9,359.6,Boleto Bancário,Cancelado,1
3,1137,2025-08-01,45,Fernanda Freitas,ucorreia@example.com,NaN,41,Olinda,PE,Roupas,Vestido Casual,3,159.9,479.7,Boleto Bancário,Em Transporte,1
4,1110,2023-08-27,149,Cecilia Novaes,ryanborges@example.org,(85) 94156-8143,29,Petrópolis,RJ,Roupas,Calça Jeans,2,129.9,259.8,Cartão De Crédito,Cancelado,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,1007,2024-12-25,109,Esther Freitas,novaesdavi-lucas@example.net,(85) 93142-9713,67,Santos,SP,Roupa,Jaqueta CortaVento,3,199.9,599.7,Cartão De Débito,Devolvido,4
500,1221,2024-03-17,265,Luiz Felipe Melo,nascimentopedro@example.net,(11) 97344-8413,59,Nan,RS,Eletrônicos,Notebook,2,3299.0,6598.0,Boleto Bancário,Devolvido,5
501,1462,2025-08-22,139,Davi Miguel Freitas,guerrarenan@example.org,(79) 99027-9526,36,Duque De Caxias,RJ,Roupas,Vestido Casual,2,159.9,319.8,Pix,Cancelado,5
502,1300,2024-02-24,319,NaN,ravi-luccadas-neves@example.net,(71) 96635-4587,45,Parintins,Amazonas,Casa E Decoração,Tapete Sala,4,249.9,999.6,Pix,Processando,4


In [17]:
#padronizando os registros de estado
df['estado'] = df['estado'].str.upper()

mapa_estados = {
    'MINAS GERAIS': 'MG',
    'CEARÁ': 'CE', 'CEARA': 'CE',
    'BAHIA': 'BA',
    'PERNAMBUCO': 'PE',
    'RIO GRANDE DO SUL': 'RS',
    'AMAZONAS': 'AM',
    'RIO DE JANEIRO': 'RJ',
    'PARANÁ': 'PR', 'PARANA': 'PR',
    'SANTA CATARINA': 'SC',
    'SÃO PAULO': 'SP', 'SAO PAULO': 'SP',
    'SERGIPE': 'SE',
    'DISTRITO FEDERAL': 'DF',
    'NÃO INFORMADO': np.nan, 'NAO INFORMADO': np.nan, '?': np.nan,
}

df['estado'] = df['estado'].replace(mapa_estados)
df

,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente
0,1041,2023-07-26,306,Renan Ribeiro,leonardo88@example.org,NaN,59,Florianópolis,SC,Roupas,Camiseta Básica,2,49.9,99.8,Cartão De Débito,Devolvido,5
1,1047,2024-03-27,195,Diego Melo,novaisdavi@example.com,(11) 97309-1356,70,Contagem,MG,Roupas,Calça Jeans,4,129.9,519.6,Cartão De Débito,Processando,10
2,1344,2023-10-16,165,Nicole Moreira,liviacaldeira@example.com,NaN,49,Porto Alegre,RS,Esporte E Lazer,Bola De Futebol Oficial,4,89.9,359.6,Boleto Bancário,Cancelado,1
3,1137,2025-08-01,45,Fernanda Freitas,ucorreia@example.com,NaN,41,Olinda,PE,Roupas,Vestido Casual,3,159.9,479.7,Boleto Bancário,Em Transporte,1
4,1110,2023-08-27,149,Cecilia Novaes,ryanborges@example.org,(85) 94156-8143,29,Petrópolis,RJ,Roupas,Calça Jeans,2,129.9,259.8,Cartão De Crédito,Cancelado,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,1007,2024-12-25,109,Esther Freitas,novaesdavi-lucas@example.net,(85) 93142-9713,67,Santos,SP,Roupa,Jaqueta CortaVento,3,199.9,599.7,Cartão De Débito,Devolvido,4
500,1221,2024-03-17,265,Luiz Felipe Melo,nascimentopedro@example.net,(11) 97344-8413,59,Nan,RS,Eletrônicos,Notebook,2,3299.0,6598.0,Boleto Bancário,Devolvido,5
501,1462,2025-08-22,139,Davi Miguel Freitas,guerrarenan@example.org,(79) 99027-9526,36,Duque De Caxias,RJ,Roupas,Vestido Casual,2,159.9,319.8,Pix,Cancelado,5
502,1300,2024-02-24,319,NaN,ravi-luccadas-neves@example.net,(71) 96635-4587,45,Parintins,AM,Casa E Decoração,Tapete Sala,4,249.9,999.6,Pix,Processando,4


In [ ]:
#verificando a existencia de duplicatas
df.duplicated().sum()
df[df.duplicated(keep=False)].sort_values('id_cliente')


,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente


In [ ]:
#Pela regra de negócio um id_cliente pode fazer vários pedidos, se eu simplesmente usasse o drop_duplicates acabaria por perder registros válidos por isso é necessário o uso do subset
df = df.drop_duplicates(subset=['id_cliente', 'id_pedido'],keep=False)
df=df.fillna('Não Informado')
df.describe()

,id_pedido,data_pedido,idade,quantidade,preco_unitario,valor_total,avaliacao_cliente
count,480.000000,480,480.0,480.0,480.000000,480.000000,480.0
mean,1240.500000,2024-08-01 05:15:00.000000256,44.733333,2.254167,493.200312,1175.845438,3.7875
min,1001.000000,2023-01-05 00:00:00,18.0,0.0,29.900000,0.000000,0.0
25%,1120.750000,2023-11-27 06:00:00,32.0,1.0,79.900000,99.875000,3.0
50%,1240.500000,2024-07-22 00:00:00,44.0,2.0,149.900000,289.900000,4.0
75%,1360.250000,2025-05-02 06:00:00,57.0,3.0,289.900000,999.000000,4.0
max,1480.000000,2025-12-27 00:00:00,75.0,5.0,3299.000000,16495.000000,10.0
std,138.708327,NaN,15.755694,1.31078,770.237813,2286.390318,1.421814


In [ ]:
(df.isnull().sum(),
df.duplicated().sum())

(id_pedido            0
 data_pedido          0
 id_cliente           0
 nome_cliente         0
 email                0
 telefone             0
 idade                0
 cidade               0
 estado               0
 categoria_produto    0
 produto              0
 quantidade           0
 preco_unitario       0
 valor_total          0
 forma_pagamento      0
 status_pedido        0
 avaliacao_cliente    0
 dtype: int64,
 np.int64(0))

## Exportação


In [21]:
df.to_csv('ecommerce_limpo.csv', index=False)